# Circular Fade for RandomEmbed

Demonstrates the `fade_radius` and `p_fade` parameters of `RandomEmbed`,
which apply a circular mask with Gaussian falloff to smoothly blend the
embedded image into the background.

**Key parameters:**
- `fade_radius=(inner, outer)`: fractions of `min(H, W)` of the input image
  - Pixels within `inner` are fully opaque (1.0)
  - Pixels beyond `outer` are fully transparent (0.0)
  - Between: Gaussian falloff (3-sigma rule)
  - `None` (default): hard rectangular boundary
- `p_fade`: probability of applying the fade per image (default 1.0)

In [ ]:
LITDATA_VAL_PATH = "s3://visionlab-datasets/imagenet1k/pre-processed/s256-l512-jpgbytes-q100-streaming/val/"

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from slipstream import (
    SlipstreamDataset,
    SlipstreamLoader,
    DecodeRandomResizeShortCropLong,
    RandomEmbed,
    ToTorchImage,
    IMAGENET_MEAN,
    IMAGENET_STD,
)

dataset = SlipstreamDataset(
    remote_dir=LITDATA_VAL_PATH,
    decode_images=False,
)
print(f"Dataset: {len(dataset):,} samples")

In [ ]:
def load_images(dataset, size=96, batch_size=8):
    """Load a batch of images decoded at the given size."""
    dec = DecodeRandomResizeShortCropLong(size=size, to_tensor=True, permute=True)
    loader = SlipstreamLoader(
        dataset, batch_size=batch_size, shuffle=False,
        pipelines={'image': [dec]},
        exclude_fields=['path'], verbose=False,
    )
    batch = next(iter(loader))
    loader.shutdown()
    return batch['image'].float() / 255.0


def show_embedded(images, n_cols=8, suptitle=None, row_labels=None,
                  mean=None, std=None):
    """Show a batch or list of batches of embedded images."""
    if isinstance(images, torch.Tensor) and images.ndim == 4:
        images = [images]
    n_rows = len(images)
    n_cols = min(n_cols, images[0].shape[0])
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5 * n_cols, 2.8 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]
    for r, batch in enumerate(images):
        for c in range(n_cols):
            img = batch[c].clone()
            if mean is not None and std is not None:
                for ch in range(img.shape[0]):
                    img[ch] = img[ch] * std[ch] + mean[ch]
            img = img.permute(1, 2, 0).clamp(0, 1).numpy()
            axes[r, c].imshow(img)
            axes[r, c].axis('off')
        if row_labels:
            axes[r, 0].set_ylabel(row_labels[r], fontsize=10, rotation=0,
                                   labelpad=60, va='center')
    if suptitle:
        fig.suptitle(suptitle, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


# Load reference batch
images = load_images(dataset, size=96, batch_size=8)
print(f"Loaded images: {images.shape}, range=[{images.min():.2f}, {images.max():.2f}]")

## 1. No Fade vs Circular Fade

Compare the default hard rectangular boundary with a circular fade.

In [ ]:
no_fade = RandomEmbed(canvas_size=224, background='power_law', alpha_range=1.5, seed=42)
with_fade = RandomEmbed(canvas_size=224, background='power_law', alpha_range=1.5, seed=42,
                        fade_radius=(0.475, 0.50), p_fade=1.0)

show_embedded(
    [no_fade(images), with_fade(images)],
    row_labels=['no fade', 'fade (0.475, 0.50)'],
    suptitle='Hard boundary vs circular fade',
)

## 2. Sharp vs Wide Fade

A small gap between inner and outer produces a sharp circle crop.
A large gap produces a gradual vignette.

In [ ]:
configs = [
    ('sharp (0.48, 0.50)', (0.48, 0.50)),
    ('medium (0.35, 0.50)', (0.35, 0.50)),
    ('wide (0.15, 0.50)', (0.15, 0.50)),
    ('funky (0.47, 0.57)', (0.47, 0.54)),
]

rows = []
labels = []
for label, fr in configs:
    embed = RandomEmbed(canvas_size=224, background='power_law',
                        alpha_range=1.5, seed=42, fade_radius=fr, p_fade=1.0)
    rows.append(embed(images))
    labels.append(label)

show_embedded(rows, row_labels=labels,
              suptitle='Fade sharpness: narrow vs wide transition band')

## 3. Inner Radius Sweep

Fix outer=0.50 and sweep inner from 0.10 (wide fade) to 0.48 (sharp circle).

In [ ]:
inners = [0.10, 0.20, 0.30, 0.40, 0.48]
rows = []
labels = []
for inner in inners:
    embed = RandomEmbed(canvas_size=224, background='power_law',
                        alpha_range=1.5, seed=42,
                        fade_radius=(inner, 0.50), p_fade=1.0)
    rows.append(embed(images))
    labels.append(f'inner={inner}')

show_embedded(rows, row_labels=labels,
              suptitle='Inner radius sweep (outer=0.50 fixed)')

## 4. Fade with Different Backgrounds

Circular fade combined with each background type.

In [ ]:
bg_configs = [
    ('zeros', dict(background='zeros')),
    ('mean (ImageNet)', dict(background='mean', mean=[0.485, 0.456, 0.406])),
    ('power_law (\u03b1=1.5)', dict(background='power_law', alpha_range=1.5, seed=42)),
]

rows = []
labels = []
for label, kwargs in bg_configs:
    embed = RandomEmbed(canvas_size=224, fade_radius=(0.35, 0.50), p_fade=1.0, **kwargs)
    rows.append(embed(images))
    labels.append(label)

show_embedded(rows, row_labels=labels,
              suptitle='Circular fade with different backgrounds')

## 5. Random Positions with Circular Fade

Circular fade combined with random positioning.

In [ ]:
seeds = [42, 43, 44]
rows = []
labels = []
for seed in seeds:
    embed = RandomEmbed(
        canvas_size=224,
        x_range=(0, 1), y_range=(0, 1),
        background='power_law', alpha_range=1.5,
        fade_radius=(0.35, 0.50), p_fade=1.0,
        seed=seed,
    )
    rows.append(embed(images))
    labels.append(f'seed={seed}')

show_embedded(rows, row_labels=labels,
              suptitle='Random positions + circular fade (power_law background)')

## 6. Visualise the Fade Mask

Show the actual mask shape for different `fade_radius` settings.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
fade_configs = [
    ('(0.48, 0.50)', (0.48, 0.50)),
    ('(0.40, 0.50)', (0.40, 0.50)),
    ('(0.25, 0.50)', (0.25, 0.50)),
    ('(0.10, 0.50)', (0.10, 0.50)),
]
for ax, (label, fr) in zip(axes, fade_configs):
    embed = RandomEmbed(canvas_size=224, fade_radius=fr)
    mask = embed._make_fade_mask(96, 96, 'cpu', torch.float32)
    ax.imshow(mask[0, 0].numpy(), cmap='gray', vmin=0, vmax=1)
    ax.set_title(label, fontsize=10)
    ax.axis('off')
fig.suptitle('Fade masks (96\u00d796 image)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Stochastic Fade (`p_fade`)

`p_fade` controls the probability of applying the circular fade **per image**.
At `p_fade=0.5`, roughly half the images get the circular fade and half keep
the hard rectangular boundary — preventing a model from relying on a single
fixed boundary shape.

In [ ]:
# p_fade=1.0 (always) vs p_fade=0.5 (stochastic) vs p_fade=0.0 (never)
p_configs = [1.0, 0.5, 0.0]
rows = []
labels = []
for p in p_configs:
    embed = RandomEmbed(
        canvas_size=224,
        x_range=(0, 1), y_range=(0, 1),
        background='power_law', alpha_range=1.5,
        fade_radius=(0.35, 0.50), p_fade=p,
        seed=42,
    )
    rows.append(embed(images))
    labels.append(f'p_fade={p}')

show_embedded(rows, row_labels=labels,
              suptitle='Stochastic fade: per-image probability of circular mask')

## Summary

| Parameter | Effect |
|-----------|--------|
| `fade_radius=None` | Hard rectangular boundary (default) |
| `fade_radius=(0.48, 0.50)` | Sharp circle crop (thin 2% fade band) |
| `fade_radius=(0.35, 0.50)` | Medium fade (15% transition) |
| `fade_radius=(0.10, 0.50)` | Wide vignette (40% gradual falloff) |
| `p_fade=1.0` | Always apply fade (default) |
| `p_fade=0.5` | Stochastic: ~half faded, ~half rectangular |
| `p_fade=0.0` | Never apply fade (hard boundary) |

Both `inner` and `outer` are fractions of `min(H, W)` of the input image.
The Gaussian falloff uses the 3-sigma rule: `sigma = (outer - inner) / 3`.